# Monosphere プリセット分析

Drive `monosphere` の書き出し済みプリセット83件（public 28 / frontier 41 / showcase 14）を対象に、
重複・カバレッジ（発生源×運動タイプ）・パラメータ統計・類似度・PCA を計算します。
結果は `preset_analysis_results.json` に保存されます。

In [ ]:
# Monosphere プリセット分析ノートブック（2026-11-09、Muse が Colab MCP 経由で実行した分析の再現版）
# 使い方: monosphere フォルダ内の3つの書き出しJSON（public-library.json /
# frontier-presets-20261107.json / showcase-presets-20261108.json）を
# このノートブックと同じ場所に置いて、上から実行してください。
import json
from collections import Counter
import numpy as np

def load(fn):
    d = json.load(open(fn))
    return [{"id": p["id"], "name": p["name"], "config": p["config"]} for p in d["presets"]]

datasets = {
    "public": load("public-library.json"),
    "frontier": load("frontier-presets-20261107.json"),
    "showcase": load("showcase-presets-20261108.json"),
}


In [ ]:


# ---------- census ----------
all_presets = []  # (dataset, preset)
for ds, ps in datasets.items():
    for p in ps:
        all_presets.append((ds, p))
ids = Counter(p["id"] for _, p in all_presets)
names = Counter(p["name"] for _, p in all_presets)
census = {
    "per_dataset": {ds: len(ps) for ds, ps in datasets.items()},
    "total": len(all_presets),
    "unique_ids": len(ids),
    "duplicate_ids": {k: v for k, v in ids.items() if v > 1},
    "duplicate_names": {k: v for k, v in names.items() if v > 1},
    "avg_config_keys": {ds: round(float(np.mean([len(p["config"]) for p in ps])), 1)
                        for ds, ps in datasets.items()},
    "key_universe": {ds: len(set().union(*[set(p["config"]) for p in ps])) if ps else 0
                     for ds, ps in datasets.items()},
}

# ---------- enum coverage ----------
ENUM_FIELDS = ["layer2Source", "layer2Type", "layer3Source", "layer3Type",
               "layer2TemporalProfile", "layer3TemporalProfile",
               "layer2MaterialStyle", "layer3MaterialStyle",
               "gpgpuAttractorType", "gpgpuVFieldType", "gpgpuEmitShape",
               "postFxStackProfile", "synthWaveform", "synthScale", "viewMode"]
coverage = {}
for f in ENUM_FIELDS:
    c = Counter()
    carrying = 0
    for _, p in all_presets:
        if f in p["config"]:
            carrying += 1
            c[str(p["config"][f])] += 1
    coverage[f] = {"carrying": carrying, "distinct": len(c),
                   "values": dict(c.most_common())}

# ---------- source x type matrices ----------
def matrix(layer):
    sf, tf = f"{layer}Source", f"{layer}Type"
    sources, types, combos = set(), set(), Counter()
    for _, p in all_presets:
        cfg = p["config"]
        if sf in cfg and tf in cfg:
            sources.add(str(cfg[sf])); types.add(str(cfg[tf]))
            combos[(str(cfg[sf]), str(cfg[tf]))] += 1
    missing = [(s, t) for s in sorted(sources) for t in sorted(types)
               if (s, t) not in combos]
    return {"sources": sorted(sources), "types_count": len(types),
            "combos_present": len(combos),
            "grid_size": len(sources) * len(types),
            "missing_count": len(missing),
            "missing_examples": missing[:20],
            "top_combos": [(f"{s} x {t}", n) for (s, t), n in combos.most_common(8)]}

matrices = {"layer2": matrix("layer2"), "layer3": matrix("layer3")}

# ---------- text map strings ----------
texts = Counter()
for _, p in all_presets:
    for f in ("layer2TextMapText", "layer3TextMapText"):
        v = p["config"].get(f)
        if isinstance(v, str) and v:
            texts[v] += 1

# ---------- numeric stats + similarity on full-config presets ----------
full = [p for ds, p in all_presets if ds in ("frontier", "showcase")]
shared = set(full[0]["config"])
for p in full[1:]:
    shared &= set(p["config"])
num_keys = sorted(k for k in shared
                  if isinstance(full[0]["config"][k], (int, float))
                  and not isinstance(full[0]["config"][k], bool))
bool_keys = sorted(k for k in shared if isinstance(full[0]["config"][k], bool))
X = np.array([[float(p["config"][k]) for k in num_keys] for p in full], dtype=float)
stds = X.std(axis=0)
const_keys = [num_keys[i] for i in range(len(num_keys)) if stds[i] == 0]
means = X.mean(axis=0)
cv = np.array([stds[i] / abs(means[i]) if means[i] != 0 else (stds[i] if stds[i] > 0 else 0.0)
               for i in range(len(num_keys))])
top_var = sorted(range(len(num_keys)), key=lambda i: -cv[i])[:15]
bool_on = {k: round(float(np.mean([1.0 if p["config"][k] else 0.0 for p in full])), 2)
           for k in bool_keys}
numeric_stats = {
    "n_full_presets": len(full),
    "shared_keys": len(shared), "numeric_keys": len(num_keys),
    "constant_numeric_keys": len(const_keys), "constant_examples": const_keys[:25],
    "top_variable": [{"key": num_keys[i], "mean": round(float(means[i]), 3),
                      "std": round(float(stds[i]), 3),
                      "min": round(float(X[:, i].min()), 3),
                      "max": round(float(X[:, i].max()), 3)} for i in top_var],
    "bool_keys": len(bool_keys),
    "bool_always_on": [k for k, v in bool_on.items() if v == 1.0],
    "bool_always_off": [k for k, v in bool_on.items() if v == 0.0],
}

# similarity (z-scored on varying keys)
keep = [i for i in range(len(num_keys)) if stds[i] > 0]
Z = (X[:, keep] - means[keep]) / stds[keep]
norms = np.linalg.norm(Z, axis=1, keepdims=True)
Zn = Z / np.where(norms == 0, 1, norms)
S = Zn @ Zn.T
np.fill_diagonal(S, -2)
nn_idx = S.argmax(axis=1)
nn_val = S.max(axis=1)
pairs = []
seen = set()
for i in range(len(full)):
    j = int(nn_idx[i])
    key = tuple(sorted((i, j)))
    if key not in seen:
        seen.add(key)
        pairs.append((float(S[i, j]), full[i]["name"], full[j]["name"]))
pairs.sort(reverse=True)
isolated = sorted(range(len(full)), key=lambda i: nn_val[i])[:5]
similarity = {
    "most_similar_pairs": [{"a": a, "b": b, "cosine": round(s, 4)} for s, a, b in pairs[:8]],
    "most_isolated": [{"name": full[i]["name"], "nearest": full[int(nn_idx[i])]["name"],
                       "cosine": round(float(nn_val[i]), 4)} for i in isolated],
    "mean_nearest_cosine": round(float(nn_val.mean()), 4),
}

# PCA
Zc = Z - Z.mean(axis=0)
U, Sv, Vt = np.linalg.svd(Zc, full_matrices=False)
var = (Sv ** 2) / (len(full) - 1)
evr = var / var.sum()
coords = (U[:, :2] * Sv[:2])
pca = {
    "pc1_explained": round(float(evr[0]), 4), "pc2_explained": round(float(evr[1]), 4),
    "top_pc1_loadings": [{"key": num_keys[keep[i]], "loading": round(float(Vt[0, i]), 3)}
                         for i in np.argsort(-np.abs(Vt[0]))[:10]],
    "coords": [{"name": full[i]["name"], "pc1": round(float(coords[i, 0]), 2),
                "pc2": round(float(coords[i, 1]), 2)} for i in range(len(full))],
    "pc1_extremes": {"high": full[int(np.argmax(coords[:, 0]))]["name"],
                     "low": full[int(np.argmin(coords[:, 0]))]["name"]},
}

results = {"census": census, "coverage": coverage, "matrices": matrices,
           "texts": dict(texts.most_common()), "numeric_stats": numeric_stats,
           "similarity": similarity, "pca": pca}
with open("preset_analysis_results.json", "w") as fh:
    json.dump(results, fh, ensure_ascii=False)

print("CENSUS", json.dumps(census, ensure_ascii=False))
print("L2 types distinct:", coverage["layer2Type"]["distinct"], "| L2 sources:", coverage["layer2Source"]["distinct"])
print("L2 matrix:", matrices["layer2"]["combos_present"], "present of", matrices["layer2"]["grid_size"], "grid; missing", matrices["layer2"]["missing_count"])
print("L3 matrix:", matrices["layer3"]["combos_present"], "present of", matrices["layer3"]["grid_size"], "grid; missing", matrices["layer3"]["missing_count"])
print("NUMERIC constant keys:", numeric_stats["constant_numeric_keys"], "of", numeric_stats["numeric_keys"])
print("SIM mean nearest cosine:", similarity["mean_nearest_cosine"])
print("PCA explained PC1/PC2:", pca["pc1_explained"], pca["pc2_explained"])
print("DONE")
